# RQ1: image/option reliance and intervention validity

Enable **Internet** and **T4 ×2**. Use the original locked runtime and the existing **rq1-inputs-v1.zip** (or its extracted Kaggle Dataset). Push the new files to GitHub first. No new model, inputs ZIP or dependency versions are required.

This bounded diagnostic runs 72 MMStar image/option trials + 6 reference repeats, then six fixed questions × 32 patches × three interventions. It loads the model once in `MODE = "all"`. Passing means the execution and numerical controls passed; it does not mean every question is image-grounded or every interpretation is confirmed.

Import into an editable draft. All execution is disabled by default, including on automatic import. Set `RUN_VALIDATION = True`, then run the code cells in order. Read `research/RQ1_VALIDITY_CHECKS.md` for the case panel, exact protocol and existing-session commands.


In [ ]:
import json
import subprocess
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_VALIDATION = False  # Change to True when ready to run manually.
MODE = "all"            # all, shortcuts, or perturbations; all loads the model once.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None         # Optional exact ZIP path or extracted bundle directory.
CHECKPOINT_ARGS = []     # Optional --model / --vision-model local original-format snapshots.

if not RUN_VALIDATION:
    raise RuntimeError("Set RUN_VALIDATION=True, enable Internet + T4 x2, then run cells manually.")
if MODE not in {"all", "shortcuts", "perturbations"}:
    raise ValueError("Unsupported MODE")
print("Preparing code; inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True)
if not (repo / "scripts/run_rq1_validation.py").is_file():
    raise RuntimeError("Push the new diagnostic files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
inputs = Path("/kaggle/working/rq1-inputs-v1")
print("Repository:", repo, flush=True)


## Prepare accepted inputs and reuse the original environment

The notebook kernel is unchanged. A fresh session recreates the locked environment; an existing session reuses it. These checks run without pretrained weights. The explicit Torch import prevents the native tests from silently skipping in Kaggle.


In [ ]:
if not (inputs / "protocol.json").exists():
    input_zip = Path(INPUT_ZIP) if INPUT_ZIP else None
    if input_zip is None:
        zip_candidates = list(Path("/kaggle/input").rglob("rq1-inputs-v1.zip"))
        zip_candidates += list(Path("/kaggle/working").glob("rq1-inputs-v1.zip"))
        extracted = []
        for path in Path("/kaggle/input").rglob("protocol.json"):
            try:
                if json.loads(path.read_text()).get("version") == "rq1-input-zero-v1":
                    if all((path.parent / name / "manifest.json").is_file() for name in ["gqa", "mmstar", "textvqa"]):
                        extracted.append(path.parent)
            except (ValueError, OSError):
                pass
        if len(zip_candidates) == 1:
            input_zip = zip_candidates[0]
        elif len(extracted) == 1:
            inputs = extracted[0]  # Kaggle may unpack a ZIP uploaded as a Dataset.
        else:
            raise RuntimeError(f"Attach rq1-inputs-v1.zip or its extracted files, and set INPUT_ZIP if needed. ZIPs: {zip_candidates}; folders: {extracted}")
    if input_zip is not None and input_zip.is_dir():
        inputs = input_zip
    elif input_zip is not None:
        inputs.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(input_zip) as z:
            for item in z.infolist():
                destination = (inputs / item.filename).resolve()
                if inputs.resolve() not in destination.parents:
                    raise ValueError("Unexpected ZIP path")
            z.extractall(inputs)
print("Accepted input directory:", inputs)
print(json.loads((inputs / "protocol.json").read_text()))
subprocess.run([
    "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
    "--output", "/kaggle/working/rq1-runtime.json",
], cwd=repo, check=True)
# Explicit import prevents Torch-dependent tests silently skipping on Kaggle.
subprocess.run([str(original_python), "-I", "-c", "import torch; print(torch.__version__)"], check=True)
for test in ["research/tests/test_rq1_validation.py", "research/tests/test_rq1_validation_native.py", "research/tests/test_rq1_native.py"]:
    subprocess.run([str(original_python), "-I", test], cwd=repo, check=True)


## Run the bounded checks

Default `MODE="all"` runs both parts consecutively with one model load. Progress is printed per generation and every eight alternative-patch records. The six intervention cases have fresh baseline/target, repeated-reference, clone-only no-op, native-parity and per-method sham checks. Failures stop without retargeting or substituting cases.

This run needs only the accepted image bundle and the profile pulled from GitHub. The profile embeds the earlier selected E/I values and source hashes; you do not need to upload earlier result ZIPs or a separate TextVQA audit. Each intervention case validates its fixed reference within this diagnostic run.


In [ ]:
def run_validation(mode=MODE):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path(f"/kaggle/working/rq1-validity-{mode}-{stamp}")
    log = folder.with_suffix(".log")
    command = [str(original_python), "-I", "scripts/run_rq1_validation.py",
               "--inputs", str(inputs), "--mode", mode, "--output", str(folder),
               "--download-budget-gib", "20", *CHECKPOINT_ARGS]
    print("Run directory:", folder, flush=True)
    with log.open("w") as stream:
        process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            print("Interrupted. Preserve the log and any report/ZIP in", folder, flush=True)
            raise
    if (folder / "report.json").exists():
        report = json.loads((folder / "report.json").read_text())
        print("Status:", report["status"], "wall seconds:", round(report["wall_seconds"], 1))
        print("Shortcut counts:", report.get("shortcut_progress"))
        print("Intervention counts:", report.get("perturbation_progress"))
        if report.get("error"):
            print("Failure:", report["error"])
    print("Download:", folder.with_suffix(".zip"), "and", log)
    if status:
        raise RuntimeError("Diagnostic failed; download its ZIP and log before restarting or changing anything.")
    return folder

validation_dir = run_validation()


## Download before stopping the session

Download **both** the timestamped ZIP and `.log`. They contain the raw trials, semantic option mappings, fixed references, numerical controls, all completed patch/method scores, source hashes and a small HTML summary. A failed run still preserves its evidence ZIP when the subprocess can exit normally. The console log remains a separate companion file.

Do not rerun large baselines or discard existing examples based on these diagnostics. Send the ZIP/log for interpretation first; decoder-layer experiments follow later.


In [ ]:
from IPython.display import FileLink, display
for path in [validation_dir.with_suffix(".zip"), validation_dir.with_suffix(".log")]:
    print(path)
    display(FileLink(str(path)))
report = json.loads((validation_dir / "report.json").read_text())
print(json.dumps({key: report.get(key) for key in ["status", "shortcut_summary", "perturbation_progress"]}, indent=2))
